# ML testing new ver.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.cluster import DBSCAN
import pickle
import time

=== XGBoost Results ===
Test R^2 (log scale): 0.9903
Test R^2 (original scale): 0.9964
Test RMSE (original scale): 0.24
XGB CV R^2 (5-fold): 0.9963 ± 0.0030
Runtime: 52.04 seconds


In [ ]:

# start_time = time.time()

# # Load and prep data
# data_path = "data/relevance/merged_real_estate_listings_parsed.csv"
# filtered_df = pd.read_csv(data_path, usecols=["Price", "Area", "Bedrooms", "Toilets", "Location", "Coordinates"])
# filtered_df = filtered_df.dropna().query("Price > 0")

# # Extract District
# filtered_df["District"] = filtered_df["Location"].str.split(",").str[-1].str.strip()

# # Parse Coordinates (for clustering and Dist_to_Center)
# coords = filtered_df["Coordinates"].str.split(",", expand=True).astype(float)
# filtered_df[["Latitude", "Longitude"]] = coords

# # Hanoi bounds
# filtered_df = filtered_df[
#     (filtered_df["Latitude"].between(20.9, 21.2)) & 
#     (filtered_df["Longitude"].between(105.7, 106.0))
# ]

# # Cluster with DBSCAN (200m radius)
# coords = filtered_df[["Latitude", "Longitude"]].values
# db = DBSCAN(eps=0.000008, min_samples=2, metric='haversine', n_jobs=-1).fit(np.radians(coords))  # ~200m
# filtered_df["Cluster_ID"] = db.labels_

# # New features
# filtered_df["Total_Rooms"] = filtered_df["Bedrooms"] + filtered_df["Toilets"]
# filtered_df["Price_per_Area"] = filtered_df["Price"] / filtered_df["Area"]
# filtered_df["Dist_to_Center"] = np.sqrt(
#     (filtered_df["Latitude"] - 21.0285)**2 + (filtered_df["Longitude"] - 105.8542)**2
# )
# filtered_df["Log_Area"] = np.log(filtered_df["Area"] + 1e-6)
# cluster_means = filtered_df.groupby("Cluster_ID")["Price"].mean().to_dict()
# filtered_df["Cluster_Price_Mean"] = filtered_df["Cluster_ID"].map(cluster_means)

# # Cap outliers
# price_cap = filtered_df["Price"].quantile(0.99)
# filtered_df["Price"] = filtered_df["Price"].clip(upper=price_cap)
# filtered_df["Price_per_Area"] = filtered_df["Price"] / filtered_df["Area"]

# # Encode and build X
# district_dummies = pd.get_dummies(filtered_df["District"], prefix="Dist")
# cluster_dummies = pd.get_dummies(filtered_df["Cluster_ID"], prefix="Cluster")
# X = pd.concat([filtered_df[["Log_Area", "Total_Rooms", "Price_per_Area", "Cluster_Price_Mean"]], 
#                district_dummies, cluster_dummies], axis=1).values
# Y = np.log(filtered_df["Price"].values + 1e-6)

# # Split
# X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# # XGB: Kept lean
# xgb = XGBRegressor(
#     n_estimators=600,           # Kept at 600 for balance
#     learning_rate=0.06, 
#     max_depth=5, 
#     subsample=0.85, 
#     colsample_bytree=0.75, 
#     reg_alpha=0.1, 
#     reg_lambda=0.2, 
#     random_state=42, n_jobs=-1, tree_method='hist'
# )
# xgb.fit(X_train, Y_train)
# Y_pred_xgb = xgb.predict(X_test)

# # Eval (no ensemble, just XGB)
# print("=== XGBoost Results ===")
# r2_log = r2_score(Y_test, Y_pred_xgb)
# Y_test_exp = np.exp(Y_test)
# Y_pred_exp = np.exp(Y_pred_xgb)
# test_mse = mean_squared_error(Y_test_exp, Y_pred_exp)
# test_rmse = np.sqrt(test_mse)

# print(f"Test R^2 (log scale): {r2_log:.4f}")
# print(f"Test R^2 (original scale): {r2_score(Y_test_exp, Y_pred_exp):.4f}")
# print(f"Test RMSE (original scale): {test_rmse:.2f}")

# # Full CV
# xgb_cv = cross_val_score(xgb, X, Y, cv=5, scoring='r2', n_jobs=-1)
# print(f"XGB CV R^2 (5-fold): {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

# # Save
# xgb.save_model("xgb_model_improved.json")
# with open("test_rmse_improved.pkl", "wb") as f:
#     pickle.dump(test_rmse, f)

# print(f"Runtime: {time.time() - start_time:.2f} seconds")

=== XGBoost Results ===
Test R^2 (log scale): 0.9903
Test R^2 (original scale): 0.9964
Test RMSE (original scale): 0.24
XGB CV R^2 (5-fold): 0.9963 ± 0.0030
Runtime: 23.90 seconds


In [77]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.cluster import DBSCAN
import pickle

start_time = time.time()

# Load and prep data
data_path = "data/relevance/merged_real_estate_listings_parsed.csv"
filtered_df = pd.read_csv(data_path, usecols=["Price", "Area", "Bedrooms", "Toilets", "Location", "Coordinates"])
filtered_df = filtered_df.dropna().query("Price > 0")

# Extract District
filtered_df["District"] = filtered_df["Location"].str.split(",").str[-1].str.strip()

# Parse Coordinates
coords = filtered_df["Coordinates"].str.split(",", expand=True).astype(float)
filtered_df[["Latitude", "Longitude"]] = coords

# Hanoi bounds
filtered_df = filtered_df[
    (filtered_df["Latitude"].between(20.9, 21.2)) & 
    (filtered_df["Longitude"].between(105.7, 106.0))
]

# Cluster with DBSCAN (200m radius)
coords = filtered_df[["Latitude", "Longitude"]].values
db = DBSCAN(eps=0.000008, min_samples=2, metric='haversine', n_jobs=-1).fit(np.radians(coords))
filtered_df["Cluster_ID"] = db.labels_

# New features
filtered_df["Log_Area"] = np.log(filtered_df["Area"] + 1e-6)
cluster_means = filtered_df.groupby("Cluster_ID")["Price"].mean().to_dict()
filtered_df["Cluster_Price_Mean"] = filtered_df["Cluster_ID"].map(cluster_means)

# Cap outliers
price_cap = filtered_df["Price"].quantile(0.99)
filtered_df["Price"] = filtered_df["Price"].clip(upper=price_cap)
filtered_df["Price_per_Area"] = filtered_df["Price"] / filtered_df["Area"] * 1000

# Save metadata for app.py
district_to_cluster = filtered_df.groupby("District")["Cluster_ID"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else -1).to_dict()
district_price_per_area = filtered_df.groupby("District")["Price_per_Area"].mean().to_dict()

# Encode and build X
district_dummies = pd.get_dummies(filtered_df["District"], prefix="Dist")
X_df = pd.concat([filtered_df[["Log_Area", "Area", "Bedrooms", "Toilets", "Price_per_Area", "Cluster_Price_Mean"]], 
                  district_dummies], axis=1)
feature_names = X_df.columns.tolist()
X = X_df.values
Y = np.log(filtered_df["Price"].values + 1e-6)

# Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# XGB
xgb = XGBRegressor(
    n_estimators=600, 
    learning_rate=0.06, 
    max_depth=5, 
    subsample=0.85, 
    colsample_bytree=0.75, 
    reg_alpha=0.1, 
    reg_lambda=0.2, 
    random_state=42, 
    n_jobs=-1, 
    tree_method='hist'
)
xgb.fit(X_train, Y_train)
Y_pred_xgb = xgb.predict(X_test)

# Eval
r2_log = r2_score(Y_test, Y_pred_xgb)
Y_test_exp = np.exp(Y_test)
Y_pred_exp = np.exp(Y_pred_xgb)
test_mse = mean_squared_error(Y_test_exp, Y_pred_exp)
test_rmse = np.sqrt(test_mse)

# Full CV
xgb_cv = cross_val_score(xgb, X, Y, cv=5, scoring='r2', n_jobs=-1)

# Save the model and metadata
with open('model/xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

metadata = {
    'feature_names': feature_names,
    'district_to_cluster': district_to_cluster,
    'district_price_per_area': district_price_per_area
}
with open('model/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

In [56]:
# Print first 20 actual vs predicted prices (moved here)
print("\n=== First 20 Actual vs Predicted Prices (in VND) ===")
print(f"{'Index':<6} {'Actual Price':>15} {'Predicted Price':>15} {'Difference':>15}")
print("-" * 60)
for i in range(min(100, len(Y_test_exp))):
    actual = Y_test_exp[i]
    predicted = Y_pred_exp[i]
    difference = actual - predicted
    print(f"{i:<6} {actual:>15,.2f} {predicted:>15,.2f} {difference:>15,.2f}")



=== First 20 Actual vs Predicted Prices (in VND) ===
Index     Actual Price Predicted Price      Difference
------------------------------------------------------------
0                 7.20            7.24           -0.04
1                10.50           10.75           -0.25
2                 6.40            6.32            0.08
3                11.18           11.34           -0.16
4                 4.24            4.19            0.05
5                 6.80            6.86           -0.06
6                 3.40            3.39            0.01
7                 3.45            3.42            0.03
8                 3.40            3.42           -0.02
9                12.30           12.35           -0.05
10                4.50            4.57           -0.07
11               12.50           12.41            0.09
12                4.35            4.31            0.04
13                1.98            1.97            0.01
14                4.53            4.58           -0.05
15   

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import RobustScaler
# from sklearn.metrics import r2_score  # Added this
# import tensorflow as tf
# from tensorflow.keras import layers, models, callbacks
# import time

# start_time = time.time()

# # Load and prep data
# data_path = "data/relevance/merged_real_estate_listings_parsed.csv"
# filtered_df = pd.read_csv(data_path, usecols=["Price", "Area", "Bedrooms", "Toilets", "Location", "Coordinates"])
# filtered_df = filtered_df.query("Price > 0").fillna({"Bedrooms": filtered_df["Bedrooms"].median(), 
#                                                      "Toilets": filtered_df["Toilets"].median()})

# # Extract District
# filtered_df["District"] = filtered_df["Location"].str.split(",").str[-1].str.strip()

# # Parse Coordinates
# coords = filtered_df["Coordinates"].str.split(",", expand=True).astype(float)
# filtered_df[["Latitude", "Longitude"]] = coords

# # Apply Hanoi bounds filter
# filtered_df = filtered_df[
#     (filtered_df["Latitude"].between(20.9, 21.2)) & 
#     (filtered_df["Longitude"].between(105.7, 106.0))
# ].reset_index(drop=True)

# # Cap outliers
# price_cap = filtered_df["Price"].quantile(0.99)
# filtered_df["Price"] = filtered_df["Price"].clip(upper=price_cap)

# # Encode categorical features
# district_dummies = pd.get_dummies(filtered_df["District"], prefix="Dist", dtype=float)

# # Scale numeric features
# scaler = RobustScaler()
# numeric_cols = scaler.fit_transform(filtered_df[["Area", "Bedrooms", "Toilets", "Latitude", "Longitude"]])
# numeric_df = pd.DataFrame(numeric_cols, columns=["Scaled_Area", "Scaled_Bedrooms", "Scaled_Toilets", 
#                                                  "Scaled_Latitude", "Scaled_Longitude"])

# # Combine features
# X = pd.concat([numeric_df, district_dummies], axis=1).reset_index(drop=True).values.astype(float)
# Y = np.log(filtered_df["Price"].values + 1e-6)

# # Split data
# X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# # Define a simple NN
# model = models.Sequential([
#     layers.Input(shape=(X_train.shape[1],)),
#     layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
#     layers.BatchNormalization(),
#     layers.Dropout(0.3),
#     layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
#     layers.BatchNormalization(),
#     layers.Dropout(0.3),
#     layers.Dense(1)
# ])

# # Compile
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])

# # Callbacks
# early_stopping = callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
# lr_scheduler = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)

# # Train
# history = model.fit(
#     X_train, Y_train,
#     epochs=50,
#     batch_size=32,
#     validation_split=0.2,
#     callbacks=[early_stopping, lr_scheduler],
#     verbose=1
# )

# # Evaluate
# Y_pred = model.predict(X_test).flatten()
# Y_test_exp = np.exp(Y_test)
# Y_pred_exp = np.exp(Y_pred)

# # Metrics (using sklearn r2_score)
# r2_log = r2_score(Y_test, Y_pred)
# r2_orig = r2_score(Y_test_exp, Y_pred_exp)
# rmse_orig = np.sqrt(np.mean((Y_test_exp - Y_pred_exp)**2))

# print(f"\nNeural Network Results:")
# print(f"Test R^2 (log scale): {r2_log:.4f}")
# print(f"Test R^2 (original scale): {r2_orig:.4f}")
# print(f"Test RMSE (original scale): {rmse_orig:.2f}")

# # Save model and scaler
# model.save("nn_model_fast.h5")
# with open("scaler_nn_fast.pkl", "wb") as f:
#     pickle.dump(scaler, f)

# print(f"Runtime: {time.time() - start_time:.2f} seconds")

Epoch 1/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2.7751 - mae: 1.2697 - val_loss: 0.3169 - val_mae: 0.3643 - learning_rate: 0.0010
Epoch 2/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5903 - mae: 0.5272 - val_loss: 0.1858 - val_mae: 0.1976 - learning_rate: 0.0010
Epoch 3/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3650 - mae: 0.3922 - val_loss: 0.1803 - val_mae: 0.1927 - learning_rate: 0.0010
Epoch 4/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2917 - mae: 0.3318 - val_loss: 0.1669 - val_mae: 0.1817 - learning_rate: 0.0010
Epoch 5/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2513 - mae: 0.3012 - val_loss: 0.1643 - val_mae: 0.1852 - learning_rate: 0.0010
Epoch 6/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2226 - mae: 0.2768 - val_loss: 0.1574 - val_mae: 0.1904 - learning_rate: 0.0010
Epoch 7/50
371/371 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1983 - mae: 0.2603 - val_loss: 0.1476 - val_mae: 0.1810 - learning_rate: 0.0010
Epoch 


Neural Network Results:
Test R^2 (log scale): 0.8331
Test R^2 (original scale): 0.7189
Test RMSE (original scale): 2.23
Runtime: 45.24 seconds
